# `ordpy-gpu` — Benchmark em GPU Datacenter com Corpus Real + Sintético Completo

Este notebook executa a bateria experimental do artigo **diretamente sobre imagens reais (RAISE, 100 TIFFs)** e **imagens sintéticas (150 amostras via HuggingFace)**, baixadas e descompactadas em disco no Google Colab, extraindo os **13 canais cromáticos/espectrais** padronizados ($800\times 800$) antes da cronometragem.

**Autor:** Fábio Linhares (`fabio-linhares/ordpy-gpu`)


In [1]:
#@title 1. Instalação de dependências e verificação da GPU
!pip install -q git+https://github.com/fabio-linhares/ordpy-gpu.git ordpy opencv-python-headless huggingface_hub pillow tqdm

import cupy as cp
import torch
dev_props = cp.cuda.runtime.getDeviceProperties(0)
gpu_name = dev_props['name'].decode('utf-8')
vram_gb = dev_props['totalGlobalMem'] / (1024**3)
print(f'GPU detectada: {gpu_name} | VRAM: {vram_gb:.2f} GB')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU detectada: NVIDIA A100-SXM4-80GB | VRAM: 79.25 GB
NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07


In [2]:
#@title 2. Download paralelo do corpus real (100 RAISE .TIF) e sintético (150 imagens)
import os, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download

BASE_DIR = Path('/content/ordpy_corpus')
REAL_DIR = BASE_DIR / 'real_raise'
SYNTH_DIR = BASE_DIR / 'synthetic'
REAL_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

RAISE_URLS = [
  "http://193.205.194.113/RAISE/TIFF/r00679daet.TIF",
  "http://193.205.194.113/RAISE/TIFF/r01170470t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r01292fd8t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0141f0c9t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0149a21ct.TIF",
  "http://193.205.194.113/RAISE/TIFF/r01a5f38at.TIF",
  "http://193.205.194.113/RAISE/TIFF/r01b6aed6t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0282ca06t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r02b3a1ddt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r02b7536at.TIF",
  "http://193.205.194.113/RAISE/TIFF/r03193652t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0342f22ct.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0355b2d5t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r036e2a9at.TIF",
  "http://193.205.194.113/RAISE/TIFF/r038422cft.TIF",
  "http://193.205.194.113/RAISE/TIFF/r038922cbt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r03ed2669t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r042d7b17t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r04a210b3t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r052df821t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r056c0ee6t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r056f233bt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r06691aa5t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r068812d7t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r06e1a520t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r072649fdt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r07598392t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0773471dt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r079d086et.TIF",
  "http://193.205.194.113/RAISE/TIFF/r07cfb432t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r07df029bt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r082626aft.TIF",
  "http://193.205.194.113/RAISE/TIFF/r08b04f65t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r09245347t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r09445479t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r09851aa4t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r09a60f63t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0b03e77at.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0b13d2a2t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0b3979d0t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0b735401t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0b82ae9bt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0b99ac3ct.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0bbd0b0et.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0bf1bbect.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0c3a8ea0t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0c5e9567t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0e5dfc8et.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0e661e0dt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0f2e6baft.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0f4ece9bt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r0f86049ct.TIF",
  "http://193.205.194.113/RAISE/TIFF/r10cf67d1t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1218b33at.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1267fa5bt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r12946364t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r12cbfb1et.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1344c943t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r13928401t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r13934dc7t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r13de1e12t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r13f26c35t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r145221d9t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r14bfe6f4t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r150058a4t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r152d9ff5t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1594b4b3t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1602555bt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r16422663t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1655a008t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r17217693t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r17492cd5t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r177dd68et.TIF",
  "http://193.205.194.113/RAISE/TIFF/r17fe8532t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r185929e9t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1889220ft.TIF",
  "http://193.205.194.113/RAISE/TIFF/r18ce12dct.TIF",
  "http://193.205.194.113/RAISE/TIFF/r18d1abact.TIF",
  "http://193.205.194.113/RAISE/TIFF/r195f12adt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r198ae140t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r19fe3107t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1aa43470t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1aa4cc99t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1aa53167t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1b547547t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1b8b1b14t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1bddecd6t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1c070921t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1c24a571t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1c4d4720t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1c56a2bct.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1cad9823t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1ced1907t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1de108dft.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1e367d35t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1e8acc5ct.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1ef16e0dt.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1f1178cet.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1f1a3258t.TIF",
  "http://193.205.194.113/RAISE/TIFF/r1f2c2acft.TIF"
]

def download_raise(url):
    fname = REAL_DIR / url.split('/')[-1]
    if fname.exists() and fname.stat().st_size > 100_000:
        return True
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=45) as resp, open(fname, 'wb') as out:
            out.write(resp.read())
        return True
    except Exception as e:
        return False

print(f'Baixando {len(RAISE_URLS)} imagens reais RAISE (.TIF)...')
ok_real = 0
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(download_raise, u) for u in RAISE_URLS]
    for f in tqdm(as_completed(futs), total=len(futs)):
        if f.result():
            ok_real += 1
print(f'RAISE baixadas: {ok_real}/{len(RAISE_URLS)}')

import zipfile
zip_path = SYNTH_DIR / 'part-000001.zip'
if not zip_path.exists():
    print('Baixando DiffusionDB part-000001.zip (anonimo, sem token HF)...')
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/poloclub/diffusiondb/resolve/main/images/part-000001.zip',
        str(zip_path),
    )
ok_synth = 0
with zipfile.ZipFile(zip_path, 'r') as zf:
    img_members = [m for m in zf.namelist() if m.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))][:150]
    for member in tqdm(img_members, desc='DiffusionDB (sintéticas)'):
        out_p = SYNTH_DIR / Path(member).name
        out_p.write_bytes(zf.read(member))
        ok_synth += 1
print(f'Sintéticas baixadas: {ok_synth}/150')


Baixando 100 imagens reais RAISE (.TIF)...


  0%|          | 0/100 [00:00<?, ?it/s]

RAISE baixadas: 100/100
Baixando DiffusionDB part-000001.zip (anonimo, sem token HF)...


DiffusionDB (sintéticas):   0%|          | 0/150 [00:00<?, ?it/s]

Sintéticas baixadas: 150/150


In [3]:
#@title 3. Extração padronizada dos 13 canais (800x800) fora do cronômetro
import cv2
import numpy as np
from PIL import Image

def extract_13_channels(rgb_img_np: np.ndarray) -> np.ndarray:
    r, g, b = rgb_img_np[:, :, 0], rgb_img_np[:, :, 1], rgb_img_np[:, :, 2]
    gray = cv2.cvtColor(rgb_img_np, cv2.COLOR_RGB2GRAY)
    lum = (0.299 * r.astype(np.float32) + 0.587 * g.astype(np.float32) + 0.114 * b.astype(np.float32)).astype(np.uint8)
    hsv = cv2.cvtColor(rgb_img_np, cv2.COLOR_RGB2HSV)
    h_ch, s_hsv, v_ch = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    hls = cv2.cvtColor(rgb_img_np, cv2.COLOR_RGB2HLS)
    l_ch, s_hsl = hls[:, :, 1], hls[:, :, 2]
    ycbcr = cv2.cvtColor(rgb_img_np, cv2.COLOR_RGB2YCrCb)
    y_ch, cr_ch, cb_ch = ycbcr[:, :, 0], ycbcr[:, :, 1], ycbcr[:, :, 2]
    return np.stack([r, g, b, gray, lum, h_ch, s_hsv, v_ch, s_hsl, l_ch, y_ch, cr_ch, cb_ch], axis=0)

def load_crop_800(path: Path) -> np.ndarray:
    img = np.array(Image.open(path).convert('RGB'))
    h, w, _ = img.shape
    if h < 800 or w < 800:
        img = cv2.resize(img, (max(w, 800), max(h, 800)), interpolation=cv2.INTER_AREA)
        h, w, _ = img.shape
    cy, cx = h // 2, w // 2
    return extract_13_channels(img[cy - 400 : cy + 400, cx - 400 : cx + 400])

real_paths = sorted(REAL_DIR.glob('*.TIF'))
synth_paths = sorted([p for p in SYNTH_DIR.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}])
all_paths = real_paths + synth_paths
print(f'Preparando {len(real_paths)} reais + {len(synth_paths)} sintéticas = {len(all_paths)} imagens...')

corpus_stacks = [load_crop_800(p) for p in tqdm(all_paths)]
corpus_4d = np.stack(corpus_stacks, axis=0)  # (N_imgs, 13, 800, 800)
N_imgs, C, H_img, W_img = corpus_4d.shape
print(f'Corpus montado em RAM: {corpus_4d.shape}, {corpus_4d.nbytes / 1e9:.2f} GB')


Preparando 100 reais + 150 sintéticas = 250 imagens...


  0%|          | 0/250 [00:00<?, ?it/s]

Corpus montado em RAM: (250, 13, 800, 800), 2.08 GB


In [4]:
#@title 4. Benchmark Completo em GPU Datacenter (Corpus Real + Sintético, 2x2, 3x3, Retangulares e 4096x4096)
import json, time
import ordpy
from ordpy_gpu import complexity_entropy_batch

def bench_gpu(batch_uint8, dx, dy, reps=15):
    # Warm-up
    _ = complexity_entropy_batch(batch_uint8, dx=dx, dy=dy)
    cp.cuda.Device(0).synchronize()
    times_ms = []
    for _ in range(reps):
        t0 = time.perf_counter()
        res = complexity_entropy_batch(batch_uint8, dx=dx, dy=dy)
        cp.cuda.Device(0).synchronize()
        times_ms.append((time.perf_counter() - t0) * 1000.0)
    arr = np.array(times_ms)
    mean = float(arr.mean())
    ci95 = float(1.96 * arr.std(ddof=1) / np.sqrt(reps))
    return mean, ci95, res

results = {
    'gpu_name': gpu_name,
    'n_real': len(real_paths),
    'n_synth': len(synth_paths),
    'n_total_images': N_imgs,
    'channels_per_image': C,
    'corpus_bench': {},
    'parity_real_synth': {},
    'rectangular_bench': {},
    'stress_4096_real': {},
}

# A) Corpus inteiro (N_imgs * 13 planos de 800x800)
flat_corpus = corpus_4d.reshape(N_imgs * C, H_img, W_img)
for dx, dy in [(2, 2), (3, 3)]:
    m, ci, _ = bench_gpu(flat_corpus, dx, dy, reps=15)
    n_win = flat_corpus.shape[0] * (H_img - dy + 1) * (W_img - dx + 1)
    gwin_s = (n_win / (m / 1000.0)) / 1e9
    results['corpus_bench'][f'{dx}x{dy}'] = {
        'total_ms': m, 'ci95_ms': ci, 'ms_per_image_13ch': m / N_imgs, 'gwin_per_s': gwin_s
    }
    print(f'[Corpus {N_imgs} imgs | {dx}x{dy}] {m:.2f} ± {ci:.2f} ms ({m/N_imgs:.3f} ms/img, {gwin_s:.2f} Gwin/s)')

# B) Paridade estrita vs ordpy (1 imagem real + 1 imagem sintética, 26 planos)
pair_26 = np.concatenate([corpus_4d[0], corpus_4d[-1]], axis=0)
for dx, dy in [(2, 2), (3, 3), (2, 3)]:
    h_gpu, c_gpu = complexity_entropy_batch(pair_26, dx=dx, dy=dy)
    cpu_hc = np.array([ordpy.complexity_entropy(pair_26[i], dx=dx, dy=dy) for i in range(26)])
    dh = float(np.max(np.abs(h_gpu - cpu_hc[:, 0])))
    dc = float(np.max(np.abs(c_gpu - cpu_hc[:, 1])))
    results['parity_real_synth'][f'{dx}x{dy}'] = {'max_dH': dh, 'max_dC': dc}
    print(f'[Paridade {dx}x{dy}] max|dH|={dh:.2e}, max|dC|={dc:.2e}')

# C) Janelas retangulares sobre o par real+sintético (26 planos, 800x800)
for dx, dy in [(1, 4), (4, 1), (2, 3), (3, 2), (2, 4), (4, 2)]:
    m, ci, _ = bench_gpu(pair_26, dx, dy, reps=15)
    results['rectangular_bench'][f'{dx}x{dy}'] = {'mean_ms': m, 'ci95_ms': ci}
    print(f'[Retangular {dx}x{dy} | 26ch] {m:.3f} ± {ci:.3f} ms')

# D) Stress 4096x4096 derivado de imagem real RAISE (13 canais)
real_full = np.array(Image.open(real_paths[0]).convert('RGB'))
real_4096 = cv2.resize(real_full, (4096, 4096), interpolation=cv2.INTER_CUBIC)
stack_4096 = extract_13_channels(real_4096)
for dx, dy in [(2, 2), (3, 3)]:
    m, ci, _ = bench_gpu(stack_4096, dx, dy, reps=10)
    results['stress_4096_real'][f'{dx}x{dy}'] = {'mean_ms': m, 'ci95_ms': ci}
    print(f'[Stress 4096x4096 Real | {dx}x{dy}] {m:.2f} ± {ci:.2f} ms')

out_json = Path('/content/benchmark_datacenter_corpus.json')
out_json.write_text(json.dumps(results, indent=2))
print('Resultados salvos em:', out_json)


[Corpus 250 imgs | 2x2] 584.23 ± 1.28 ms (2.337 ms/img, 3.55 Gwin/s)
[Corpus 250 imgs | 3x3] 836.12 ± 1.00 ms (3.344 ms/img, 2.48 Gwin/s)
[Paridade 2x2] max|dH|=2.22e-16, max|dC|=1.30e-15
[Paridade 3x3] max|dH|=3.89e-15, max|dC|=1.49e-13
[Paridade 2x3] max|dH|=2.22e-16, max|dC|=1.83e-15
[Retangular 1x4 | 26ch] 13.351 ± 0.093 ms
[Retangular 4x1 | 26ch] 10.504 ± 0.382 ms
[Retangular 2x3 | 26ch] 7.853 ± 0.082 ms
[Retangular 3x2 | 26ch] 7.893 ± 0.069 ms
[Retangular 2x4 | 26ch] 6.075 ± 0.131 ms
[Retangular 4x2 | 26ch] 5.611 ± 0.088 ms
[Stress 4096x4096 Real | 2x2] 62.69 ± 0.07 ms
[Stress 4096x4096 Real | 3x3] 88.48 ± 0.39 ms
Resultados salvos em: /content/benchmark_datacenter_corpus.json
